# Imports

In [3]:
#Base imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Preprocessing imports
from imputers import MasterImputer
from sklearn.preprocessing import StandardScaler

# Modeling imports
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression



# Exploratory Data Analysis

In [6]:
data = pd.read_csv('../data/telecoms.csv')
X = data.drop(columns=['Attrition'])
y = data['Attrition']


In [7]:
X.describe()

,User ID,Mobile No.,Duration,Voicemail_Messages,Day_Minutes,Day_Calls,Day_Charge,Eve_Minutes,Eve_Calls,Eve_Charge,Night_Minutes,Night_Calls,Night_Charge,Intl_Minutes,Intl_Calls,Intl_Charge,Service_Calls
count,4150.000000,4.150000e+03,4150.000000,3681.000000,4150.000000,3640.000000,4150.000000,3626.000000,4150.000000,4150.000000,3548.000000,4150.000000,4150.000000,4150.000000,4150.000000,3561.000000,4150.000000
mean,2776.621928,7.988843e+09,99.967470,7.890519,177.145687,100.010440,391.499386,199.429923,100.250120,220.259219,199.746167,99.983855,116.795822,10.206482,4.455181,35.851342,1.495904
std,1346.549853,1.157382e+09,39.816089,13.599601,51.396781,19.569463,113.586282,50.015577,19.845768,55.418443,50.492036,19.898864,29.707936,2.759975,2.436228,9.615616,1.211282
min,406.000000,6.000109e+09,1.000000,0.000000,0.000000,0.000000,0.000000,22.300000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1589.250000,6.985293e+09,73.000000,0.000000,142.825000,87.000000,315.672500,165.125000,87.000000,182.292500,165.975000,87.000000,97.110000,8.500000,3.000000,29.900000,1.000000
50%,2830.500000,7.958110e+09,100.000000,0.000000,178.400000,100.000000,394.290000,199.800000,101.000000,220.805000,199.150000,100.000000,116.740000,10.300000,4.000000,36.140000,1.000000
75%,3962.750000,9.006551e+09,127.000000,18.000000,212.100000,113.000000,468.780000,232.900000,114.000000,257.400000,233.325000,113.000000,136.467500,12.000000,6.000000,42.120000,2.000000
max,5000.000000,9.998408e+09,243.000000,52.000000,351.500000,163.000000,776.880000,361.800000,170.000000,399.750000,395.000000,175.000000,231.010000,19.700000,19.000000,69.160000,8.000000


In [8]:
X.isna().sum()

User ID                 0
Mobile No.              0
User DOB                0
User Job                0
Duration                0
Intl_Plan             471
Voicemail_Plan          0
Voicemail_Messages    469
Day_Minutes             0
Day_Calls             510
Day_Charge              0
Eve_Minutes           524
Eve_Calls               0
Eve_Charge              0
Night_Minutes         602
Night_Calls             0
Night_Charge            0
Intl_Minutes            0
Intl_Calls              0
Intl_Charge           589
Service_Calls           0
dtype: int64

Areas of Note:
- 469 people got N/A as voicemail messages. This needs to be cleaned to most likely say 0 because these people did not have a voicemail plan. These 2 variables are likely to be highly correlated.
- 471 people have N/A on if they have an international plan (yes/no)
- 510 people have N/A as daycalls
- 524 people have N/A as eve_minutes
- 602 people have N/A as night_minutes
- 589 peopl have N/A for international charges


Somethings to clean data 

- if N/A for international charges and have no international plan -> 0, otherwise 1

- if N/A international plan and have international charges then plan ->yes otherwise no

- for the day calls find median time for day_minutes and divide the minites spent by that to find number of calls
    for eve_minutes and night_minutes do the opposite way

- if has voicemail plan -> median number of voicemessages, otherwise 0.




Column selection:

- **User ID** should be dropped for the model
- **User DOB** doesnt matter as much as birth year. The year is going to keep getting newer for new future users. Preventing model degredation wont really matter for this specific variable because the model will be rerun more than atleast once a year
- **Mobile No.** is not importanat but the area code is

- **Duration** can be dropped since its just the combination of all the _Minutes or atleast should be

- Everything else should just be scaled


In [9]:
(X[X["User Job"].duplicated(keep=False)]).describe()

,User ID,Mobile No.,Duration,Voicemail_Messages,Day_Minutes,Day_Calls,Day_Charge,Eve_Minutes,Eve_Calls,Eve_Charge,Night_Minutes,Night_Calls,Night_Charge,Intl_Minutes,Intl_Calls,Intl_Charge,Service_Calls
count,4140.000000,4.140000e+03,4140.000000,3672.000000,4140.000000,3631.000000,4140.000000,3616.000000,4140.000000,4140.000000,3538.000000,4140.000000,4140.000000,4140.000000,4140.000000,3553.000000,4140.000000
mean,2775.965217,7.988147e+09,99.943478,7.875817,177.130242,99.967227,391.465232,199.422262,100.247101,220.251527,199.755257,99.977295,116.800164,10.201473,4.455797,35.832398,1.495169
std,1346.870957,1.157634e+09,39.835221,13.598850,51.412957,19.549296,113.622049,50.003406,19.853560,55.407051,50.500127,19.912953,29.712423,2.759667,2.436679,9.612807,1.211097
min,406.000000,6.000109e+09,1.000000,0.000000,0.000000,0.000000,0.000000,22.300000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1587.750000,6.983676e+09,73.000000,0.000000,142.800000,87.000000,315.640000,165.175000,87.000000,182.357500,165.925000,87.000000,97.110000,8.500000,3.000000,29.900000,1.000000
50%,2828.500000,7.955418e+09,100.000000,0.000000,178.400000,100.000000,394.290000,199.800000,101.000000,220.740000,199.250000,100.000000,116.740000,10.300000,4.000000,36.140000,1.000000
75%,3962.250000,9.007061e+09,127.000000,17.250000,212.100000,113.000000,468.780000,232.825000,114.000000,257.302500,233.375000,113.000000,136.500000,12.000000,6.000000,42.120000,2.000000
max,5000.000000,9.998408e+09,243.000000,52.000000,351.500000,163.000000,776.880000,361.800000,170.000000,399.750000,395.000000,175.000000,231.010000,19.700000,19.000000,69.160000,8.000000


Only 10 people have the same job description. 
<br>At a later stage for potential improved performance we can change the users to fields from there position. for example engineer, manager etc.

In [10]:
Area_Code = X['Mobile No.'].astype(str).str[:3]
len(Area_Code.unique())

400

Theres still 400 area codes. Since people can keep their phone numbers even after leaving a state it is only a proxy for location. For these reasons it might not be a good predictor. Additionally treating it as a categorical variable could lead to overfitting since it needs to be encodied using 400 seperate variables. For this reason the initial model will be trained without it.

# FULL PREPROCESSING PIPELINE

In [11]:
def preprocessing_pipeline(X,drop_area_code=True): 
    ### Data Imputation
    imputer = MasterImputer(voicemail_strategy='zero', minute_strategy='mean', charge_strategy='mean')

    X = imputer.fit_transform(X)

    ### Data Cleaning
    X = X.drop('User ID', axis = 1)
    X = X.drop(['User Job','Duration'], axis = 1)

    X['Mobile No.'] = X['Mobile No.'].astype(str).str[:3]
    X['User DOB'] = X['User DOB'].astype(str).str[:4]

    X.rename(columns={
                    'Mobile No.': 'Area_Code',
                    'User DOB': 'Birth_Year'
                    }, inplace=True)

    X["Voicemail_Plan"] = X["Voicemail_Plan"] == "Yes"
    X["Intl_Plan"] = X["Intl_Plan"] == "Yes"
    
    if drop_area_code:
        X = X.drop('Area_Code', axis=1)

    ### Data Scaling
    numerical_features = X.columns.drop(['Intl_Plan', 'Voicemail_Plan'])

    std_scaler = StandardScaler()

    X_scaled = pd.DataFrame(
        std_scaler.fit_transform(X[numerical_features]),
        columns=numerical_features,
        index=X.index
    ).join(X[['Intl_Plan', 'Voicemail_Plan']])

    return X_scaled


In [16]:
# data preprocessing
data = pd.read_csv('../data/telecoms.csv')
X = data.drop(columns=['Attrition'])
y = data['Attrition']
X_scaled = preprocessing_pipeline(X)

# Model Selection and Evaluation

In [17]:
# Binary Classifier
sgd_clf = SGDClassifier(random_state=42)
score = cross_val_score(sgd_clf, X_scaled, y, cv=3)
float(score.mean())

0.9493974103377512

In [ ]:
# Logistic Regression
log_clf = LogisticRegression(max_iter=1000, random_state=42)
score = cross_val_score(log_clf, X_scaled, y, cv=3)
float(score.mean())

0.9513254116529227

Both have good accuray scores with the logistic regression having a better accuracy score. To ensure the classifier is behaving correctly look at the confusion matrix:

In [ ]:
from sklearn.metrics import confusion_matrix
y_pred = log_clf.fit(X_scaled, y).predict(X_scaled)
confusion_matrix(y, y_pred)

array([[3940,    3],
       [ 201,    6]])

Clearly the models are not doing as well as anticipated. They are instead just guessing the majority class. 
This is likely due to the fact that the dataset is imbalanced, with a majority of the samples belonging to the "No" class.


In [20]:
# Using stratification for the modelling

# Binary Classifier
sgd_clf = SGDClassifier(random_state=42,class_weight='balanced')
score = cross_val_score(sgd_clf, X_scaled, y, cv=3)
print("Binary classifier score",float(score.mean()))

# Logistic Regression
log_clf = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
score = cross_val_score(log_clf, X_scaled, y, cv=3)
print("Logistic Regression score",float(score.mean()))


Binary classifier score 0.7031389972094398
Logistic Regression score 0.7303588022463802


In [21]:
#Best Model: Logistic Regression
log_clf.fit(X_scaled, y)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default 

# Evaluation

The model performs okay. More improvements by more powerful models incoming